# Part 2 — Build the multilayer graph

This notebook builds the two-aspect graph and writes the snapshot that the rest
of the series reloads. Run `01_setup` first.

Each biological layer loads in one or two calls. `from_omnipath` fetches the
signaling network. `from_sbml` reads the metabolic model. Bulk `add_edges` and
`add_vertices` add the complexes, the regulation, and the coupling.

In [2]:
import polars as pl
import requests

import annnet as an
from uc2_common import *  # noqa: F403

hek = pl.read_parquet(HEK293)
hek_genes = set(hek["Gene"])
G = an.AnnNet(directed=True)
G.history.enable(True)
G.history.snapshot("init")
G.layers.set_aspects(ASPECTS, ELEM_LAYERS)
print("aspects:", G.layers.list_aspects(), "| layers:", G.layers.list_layers())

aspects: ('mechanism', 'complex') | layers: {'mechanism': ['metabolic', 'regulatory', 'signaling'], 'complex': ['member', 'monomer']}


## Proteins and their assembly state

A protein's `complex` coordinate depends on its complex membership, so this step
reads the OmniPath complex catalogue first. It keeps the complexes whose subunits
are all HEK293-expressed. It then adds every protein at `(signaling, member)` or
`(signaling, monomer)`.

In [3]:
cpx_tsv = DATA / "omnipath_complexes.tsv"
if not cpx_tsv.exists():
    cpx_tsv.write_text(requests.get(
        "https://omnipathdb.org/complexes?databases=CORUM,ComplexPortal,hu.MAP2&fields=sources&format=tsv",
        timeout=120).text)
cpx = pl.read_csv(cpx_tsv, separator="\t", infer_schema_length=5000)

complex_specs, members = [], set()
for row in cpx.iter_rows(named=True):
    subs = (row["components_genesymbols"] or "").split("_")
    if subs == [""] or not set(subs) <= hek_genes:
        continue
    vids = [f"prot:{s}" for s in subs]
    members.update(vids)
    complex_specs.append({"edge_id": f'cpx:{row["name"]}', "edge_kind": "complex", "weight": 1.0,
                          "members": [(v, (MECH_SIGNALING, CPLX_MEMBER)) for v in vids],
                          "complex_name": row["name"]})
print(f"complexes kept: {len(complex_specs):,} | member proteins: {len(members):,}")

complexes kept: 11,075 | member proteins: 10,831


In [4]:
def prot_coord(vid):
    return (MECH_SIGNALING, CPLX_MEMBER if vid in members else CPLX_MONOMER)

rows = hek.select(vertex_id="prot:" + pl.col("Gene"), kind=pl.lit(KIND_PROTEIN),
                  gene_symbol="Gene", uniprot="Uniprot",
                  compartment=pl.col("compartment").fill_null("c"), hpa_location="hpa_location").to_dicts()
G.add_vertices([r for r in rows if r["vertex_id"] in members], layer=(MECH_SIGNALING, CPLX_MEMBER))
G.add_vertices([r for r in rows if r["vertex_id"] not in members], layer=(MECH_SIGNALING, CPLX_MONOMER))
G.history.snapshot("after_proteins")
print("member:", len(G.layers.layer_vertex_set((MECH_SIGNALING, CPLX_MEMBER))),
      "| monomer:", len(G.layers.layer_vertex_set((MECH_SIGNALING, CPLX_MONOMER))))

member: 10831 | monomer: 7726


## Signaling layer — the OmniPath network in one call

`from_omnipath` fetches the OmniPath prior-knowledge network and returns a graph.
This step takes its 85,217-edge table, keeps the pairs whose endpoints are both
HEK293-expressed, and adds each as a signed, directed edge between two protein
supra-nodes.

In [5]:
import contextlib, io
with contextlib.redirect_stdout(io.StringIO()):
    pkn = an.from_omnipath(dataset="omnipath", query={"genesymbols": True},
                        source_col="source_genesymbol", target_col="target_genesymbol",
                        edge_attr_cols=["is_stimulation", "is_inhibition", "sources"],
                        load_vertex_annotations=False)
sig = pkn.views.edges().to_pandas()
sig = sig[sig["source"].isin(hek_genes) & sig["target"].isin(hek_genes)]
G.add_edges([{"source": (f"prot:{r.source}", prot_coord(f"prot:{r.source}")),
              "target": (f"prot:{r.target}", prot_coord(f"prot:{r.target}")),
              "weight": sign_of(r.is_stimulation, r.is_inhibition), "edge_kind": "signaling",
              "is_stimulation": bool(r.is_stimulation), "is_inhibition": bool(r.is_inhibition)}
             for r in sig.itertuples(index=False)], default_edge_directed=True)
G.history.snapshot("after_signaling")
print(f"signaling edges: {len(sig):,}")

/home/l1boll/miniconda3/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


signaling edges: 31,898


## Complex hyperedges and the metabolic model

Each complex becomes one undirected hyperedge — a single incidence-matrix column,
not a clique of pairwise edges. One `from_sbml` call reads the Human-GEM model:
its reactions become signed hyperedges and its compartments become slices.
`from_sbml` adds metabolite and boundary vertices without a `kind`, so this step
tags them for the later `to_pyg` export.

In [6]:
G.add_edges(complex_specs, layer=(MECH_SIGNALING, CPLX_MEMBER))
G.history.snapshot("after_complex")

if not HUMANGEM.exists():
    HUMANGEM.write_bytes(requests.get(
        "https://raw.githubusercontent.com/SysBioChalmers/Human-GEM/main/model/Human-GEM.xml",
        timeout=600, allow_redirects=True).content)
an.from_sbml(str(HUMANGEM), graph=G, slice=MECH_METABOLIC, layer=METAB_COORD, preserve_stoichiometry=True)

V = G.views.vertices().to_pandas()
un = V[V["kind"].isna()]
G.attrs.set_vertex_attrs_bulk({v: {"kind": KIND_METABOLITE}
                               for v in un.loc[~un.vertex_id.str.startswith("__"), "vertex_id"]})
G.attrs.set_vertex_attrs_bulk({v: {"kind": "boundary"}
                               for v in un.loc[un.vertex_id.str.startswith("__"), "vertex_id"]})
G.history.snapshot("after_metabolic")
print("complex hyperedges + metabolic reactions added | total edges:", G.global_count("edges"))

complex hyperedges + metabolic reactions added | total edges: 49325


## Regulatory layer and inter-mechanism coupling

DoRothEA (confidence A and B) supplies transcription-factor→target edges between
genes. Two coupling families then bridge the `mechanism` aspect: translation
(gene→protein) and TF-anchor (protein→gene). The build adds no identity coupling;
the two-aspect design removes the need for it.

In [8]:
dor_tsv = DATA / "dorothea.tsv"
if not dor_tsv.exists():
    dor_tsv.write_text(requests.get(
        "https://omnipathdb.org/interactions?datasets=dorothea&dorothea_levels=A,B"
        "&genesymbols=yes&fields=sources,references&format=tsv", timeout=120).text)
dor = pl.read_csv(dor_tsv, separator="\t", infer_schema_length=5000).filter(
    pl.col("source_genesymbol").is_in(hek_genes) & pl.col("target_genesymbol").is_in(hek_genes))

reg_genes = sorted(set(dor["source_genesymbol"]) | set(dor["target_genesymbol"]))
G.add_vertices([{"vertex_id": f"gene:{g}", "kind": KIND_GENE, "gene_symbol": g} for g in reg_genes],
               layer=GENE_COORD)
G.add_edges([{"edge_id": f'reg:{r["source_genesymbol"]}->{r["target_genesymbol"]}',
              "source": (f'gene:{r["source_genesymbol"]}', GENE_COORD),
              "target": (f'gene:{r["target_genesymbol"]}', GENE_COORD),
              "weight": sign_of(r["is_stimulation"], r["is_inhibition"]), "edge_kind": "regulatory"}
             for r in dor.iter_rows(named=True)], default_edge_directed=True)
G.history.snapshot("after_regulatory")
print(f"regulatory genes: {len(reg_genes):,} | edges: {dor.height:,}")

regulatory genes: 5,197 | edges: 15,145


In [9]:
tfs = set(dor["source_genesymbol"])
G.add_edges([{"edge_id": f"trans:{g}", "edge_kind": "coupling_translation", "weight": 1.0,
              "source": (f"gene:{g}", GENE_COORD), "target": (f"prot:{g}", prot_coord(f"prot:{g}"))}
             for g in sorted(set(reg_genes) & hek_genes)], default_edge_directed=True)
G.add_edges([{"edge_id": f"anchor:{tf}", "edge_kind": "coupling_tf_anchor", "weight": 1.0,
              "source": (f"prot:{tf}", prot_coord(f"prot:{tf}")), "target": (f"gene:{tf}", GENE_COORD)}
             for tf in sorted(tfs & hek_genes)], default_edge_directed=True)
G.history.snapshot("after_coupling")
G.write(str(SNAPSHOT), overwrite=True)
print(f"saved {SNAPSHOT}  |  V={G.global_count('vertices'):,}  E={G.global_count('edges'):,}")

saved data/uc2.annnet  |  V=32,211  E=69,888
